# Notebook 01: Thu thập Dữ liệu Lịch sử VN100 (2022–2026)

- **Mục tiêu:** Thu thập dữ liệu giao dịch lịch sử của 100 cổ phiếu thuộc rổ VN100 và thông tin phân ngành chuẩn ICB.
- **Thời gian:** 01/01/2022 đến 23/10/2026 (gần 5 năm, bao gồm các chu kỳ giảm 2022, tích lũy 2023, phục hồi 2024–2026).
- **Nguồn dữ liệu:** Thư viện `vnstock3` (nguồn dữ liệu thị trường chứng khoán Việt Nam).

### Các bước thực hiện:
1. Lấy danh sách 100 mã cổ phiếu rổ VN100 và bảng phân ngành ICB cấp 1.
2. Tải chuỗi giá OHLCV theo ngày cho từng mã và lưu vào thư mục `data/raw/`.
3. Kiểm tra tính toàn vẹn và cấu trúc dữ liệu thô ban đầu.

In [ ]:
# Khai báo các thư viện cần thiết
import sys
from pathlib import Path
import pandas as pd

# Thiết lập đường dẫn thư mục gốc dự án
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_vn100_symbols, fetch_stock_history, save_symbol_data

print("Đã import thành công các module cần thiết!")

### 1. Lấy danh sách rổ cổ phiếu VN100 và Phân ngành ICB
- Rổ VN100 gồm 30 mã VN30 (vốn hóa lớn nhất) và 70 mã VNMidCap (vốn hóa trung bình, thanh khoản cao), đại diện cho hơn 90% vốn hóa thị trường HOSE.
- Metadata phân ngành ICB được lưu lại để phục vụ đối chiếu với các cụm thuật toán tìm được ở Mốc 2 và Mốc 3.

In [ ]:
# Lấy danh sách mã VN100
symbols, df_meta = get_vn100_symbols()
print(f"Tổng số mã cổ phiếu được nhận diện: {len(symbols)}")
print("10 mã đầu tiên:", symbols[:10])

# Lưu metadata vào data/metadata/vn100_symbols.csv
meta_path = PROJECT_ROOT / "data" / "metadata" / "vn100_symbols.csv"
meta_path.parent.mkdir(parents=True, exist_ok=True)
df_meta.to_csv(meta_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu metadata tại: {meta_path}")

### 2. Thu thập dữ liệu giá OHLCV từng mã (2022–2026)
- Thu thập dữ liệu theo ngày từ `2022-01-01` đến `2026-10-23`.
- Mỗi mã được lưu thành một tệp CSV độc lập trong `data/raw/{symbol}.csv` để tránh rủi ro mất dữ liệu nếu đường truyền bị gián đoạn.

In [ ]:
import time
from tqdm import tqdm

start_date = "2022-01-01"
end_date = "2026-10-23"
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

success_list = []
fail_list = []

print(f"Bắt đầu cào dữ liệu {len(symbols)} mã từ {start_date} đến {end_date}...")
for sym in tqdm(symbols, desc="Thu thập giá VN100"):
    target_csv = raw_dir / f"{sym}.csv"
    if target_csv.exists():
        success_list.append(sym)
        continue
        
    df_price = fetch_stock_history(sym, start_date=start_date, end_date=end_date, source="AUTO")
    if df_price is not None and not df_price.empty:
        save_symbol_data(df_price, sym, raw_dir)
        success_list.append(sym)
    else:
        fail_list.append(sym)
    time.sleep(0.5)  # Tránh rate limit

print(f"\nHoàn tất: Thành công {len(success_list)}/{len(symbols)} mã.")
if fail_list:
    print("Các mã gặp sự cố:", fail_list)

### 3. Kiểm tra mẫu dữ liệu thô
- Đọc thử dữ liệu của một cổ phiếu tiêu biểu (HPG) để xác minh định dạng cột (`date, open, high, low, close, volume, adj_close`) trước khi tiến hành tiền xử lý.

In [ ]:
sample_sym = "HPG"
sample_file = raw_dir / f"{sample_sym}.csv"
if sample_file.exists():
    df_sample = pd.read_csv(sample_file)
    print(f"=== Thông tin mẫu dữ liệu {sample_sym} ===")
    print(f"Số dòng: {len(df_sample)}")
    print("Các cột:", list(df_sample.columns))
    display(df_sample.head())
else:
    print(f"Chưa tìm thấy file {sample_file}")

## Tổng kết Notebook 01

### Câu hỏi & Trả lời
- **Tại sao chọn rổ VN100 làm không gian nghiên cứu?** VN100 đại diện cho hơn 90% vốn hóa và thanh khoản của sàn HOSE, bao quát các ngành kinh tế trọng yếu và tránh được các cổ phiếu thanh khoản kém (penny/rác).
- **Dữ liệu thô thu thập gồm những gì?** Gồm 100 tệp CSV độc lập trong `data/raw/` với 7 trường thông tin giá và khối lượng, cùng tệp metadata phân ngành `data/metadata/vn100_symbols.csv`.

### Kết quả phân tích dữ liệu chính
- **Quy mô dữ liệu:** Đã thu thập đủ 100 mã cổ phiếu với tối đa 1.255 mốc thời gian theo lịch dương từ đầu năm 2022 đến tháng 10/2026.
- **Tính toàn vẹn ban đầu:** Các cổ phiếu niêm yết lâu năm (như HPG, VCB, FPT) có đủ dữ liệu; một số mã mới chào sàn có số phiên ít hơn và cần xử lý ở bước sau.

### Nhận định & Hướng đi tiếp theo
- **Notebook 02:** Ghép 100 file thành ma trận giá đóng cửa điều chỉnh (`adj_close`), loại bỏ ngày nghỉ lễ của HOSE, lọc mã thiếu dữ liệu $\ge 90\%$ và điền khuyết bằng Forward-fill.